# Practical 02b — The Logic Detective: Propositional Logic and Model Checking (Step-by-Step Edition)

**Artificial Intelligence 3(2-1) · BS Computer Science · Semester III**

| | |
|---|---|
| **Related lectures** | Lecture 3 (Knowledge-Based Systems), Lecture 4 (Propositions and Semantics), Logical Agents (AIMA 4e §7.1–7.4) |
| **Companion** | Practical 02S (the same ideas on paper) |
| **Duration** | 3 hours |
| **Marks** | 20 (the marks for each part are in the table in Section 2) |
| **Submission** | This notebook, run top to bottom from a clean kernel |

Detective stories work like this: the detective collects **clues**, and then says *"so it must
have been Ms Scarlet, in the library, with the knife."* No guessing. Every other answer
contradicts a clue.

In this practical you will build a small program that reasons in exactly that way. You give it
facts written in **propositional logic**. It checks every possible world, throws away the
worlds that break a clue, and reports what is true in all the worlds that are left. Then you
use it to solve four cases, some of which a person finds hard, and one puzzle the program shows
cannot be solved at all.

## 0. How to use this notebook

1. **Run cells in order**, top to bottom. Click a cell and press **Shift + Enter**.
2. `___` (three underscores) in code is a **blank**. Replace it with your answer and run the
   cell again.
3. After most exercises there is a **✅ You should see** box. Compare it with what your cell
   printed. If it is the same, move on. If it is different, look at your blanks again.
4. Some cells ask you to **predict** first. Write your guess in the blank, *then* run the next
   cell to see the real answer. A wrong guess costs nothing, and it is the quickest way to learn.
5. Each part starts with a **Remember** box (the facts you must know for the exam) and ends with
   **Check yourself** questions.
6. If you restart the notebook, run every cell from the setup cell down again. Later parts use
   the code you write in earlier parts.

> 💡 **Tip.** When a blank stumps you, read the English sentence in the comment next to it out
> loud. Nearly every blank is that sentence, written in Python.

## 1. Objectives

By the end of this session you will be able to:

1. Write sentences of propositional logic as Python objects, and evaluate them in a **model**.
2. Build the truth table of ¬, ∧, ∨, ⇒ and ⇔, and explain why *false ⇒ anything* is true.
3. Decide **entailment** (KB ⊨ α) by **model checking**: α must be true in every model of the KB.
4. Answer a query as YES, NO or MAYBE, and recognise a **contradictory** knowledge base.
5. Translate English clues, including "exactly one" and "exactly two", into logic.
6. Explain why model checking costs 2ⁿ, and make a solver faster by listing the KB's models once.
7. Solve truth-teller and liar puzzles with ⇔, and build a knowledge-based agent that TELLs and ASKs.

## 2. The case files

You have joined the **AI Detective Agency**. Your first job is to build the agency's reasoning
engine. Then the cases arrive.

| Part | What you do | Source file it rebuilds | Marks |
|---|---|---|---|
| A | Build the language of logic: symbols and the five connectives | `logic.py` | 4 |
| B | Build the reasoning engine: models, entailment, YES/NO/MAYBE | `logic.py` | 4 |
| C | **Case 1: Did it rain?** Three facts, one conclusion | `harry.py` | 2 |
| D | **Case 2: The Clue mystery.** Who, where, and with what? | `clue.py` | 2 |
| E | **Case 3: The sorting puzzle.** Four people, four houses | `puzzle.py` | 2 |
| F | **Case 4: Mastermind.** Crack a four-colour code from two guesses | `mastermind.py` | 2 |
| G | **The engine is too slow.** Make it fast | (new) | 2 |
| H | **Final case: the island of truth-tellers and liars** | (new) | 2 |
| I | *Bonus, not marked:* an agent that plays Mastermind by itself | (new) | — |

**Two words you will use all afternoon:**

| Word | Meaning | Example |
|---|---|---|
| **Model** | one possible world: every symbol set to True or False | `{'rain': True, 'hagrid': False}` |
| **Knowledge base (KB)** | everything we know, joined with AND | the clues of a case |
| **Entailment, KB ⊨ α** | α is true in **every** model in which the KB is true | "it must have rained" |

## 3. Apparatus and setup

**Software:** Python 3.10 or later, in Jupyter or VS Code. Standard library only, so there is
nothing to install.

**Where the code comes from.** The engine and Cases 1–4 follow the files in `src/`
(`logic.py`, `harry.py`, `clue.py`, `puzzle.py`, `mastermind.py`), from Harvard's
*CS50's Introduction to Artificial Intelligence with Python*, Lecture 1 (Knowledge),
shared under CC BY-NC-SA 4.0. You will rebuild `logic.py` yourself in Parts A and B, a little
simplified, so you do not need the files to run this notebook.

Run the cell below **first, every time you open the notebook.**

In [ ]:
import itertools  # product(): every model; combinations(): pick k of n
import time               # to time the slow cases

___ = None  # the blank you will replace with your answers
print('Setup done. Welcome to the AI Detective Agency.')

**✅ You should see:**

```
Setup done. Welcome to the AI Detective Agency.
```

---
## Part A — The language of logic (4 marks)

> ### Remember
>
> • A **proposition symbol** stands for a statement that is either true or false: `rain` means "it is raining".
>
> • **Connectives** build bigger sentences: ¬ (not), ∧ (and), ∨ (or), ⇒ (implies), ⇔ (if and only if).
>
> • A **model** gives every symbol a value. A sentence has no truth value on its own; it is true or false **in a model**.
>
> • P ⇒ Q is false in only **one** case: P true and Q false. P ⇔ Q is true when P and Q have the **same** value.

Every sentence will be a Python object with three methods:

| Method | What it returns | Example for `And(rain, wet)` |
|---|---|---|
| `evaluate(model)` | True or False in that model | `True` if both are True in the model |
| `formula()` | the sentence as text | `'rain ∧ wet'` |
| `symbols()` | the set of symbol names inside it | `{'rain', 'wet'}` |

### A1 — Symbols and models

`Sentence` is the parent class. `Symbol` is the simplest sentence: to evaluate it, look its
name up in the model. Run the cell, then fill the blanks in the cell after it.

In [ ]:
class Sentence:
    """The parent of every logical sentence."""
    def evaluate(self, model):
        raise NotImplementedError('each sentence says how to evaluate itself')
    def formula(self):
        return ''
    def symbols(self):
        return set()
    def __repr__(self):            # so print() shows the formula
        return self.formula()


class Symbol(Sentence):
    def __init__(self, name):
        self.name = name
    def evaluate(self, model):
        return model[self.name]     # a model is a dict: name -> True/False
    def formula(self):
        return self.name
    def symbols(self):
        return {self.name}


def wrap(sentence):
    """The formula, in brackets unless it is a symbol or a 'not'."""
    text = sentence.formula()
    return text if isinstance(sentence, (Symbol, Not)) else '(' + text + ')'


rain = Symbol('rain')
print(rain, rain.symbols())

**✅ You should see:**

```
rain {'rain'}
```

**Your turn.** Make a model in which it is **not** raining, and evaluate `rain` in it.

In [ ]:
dry_day = {'rain': ___}         # a model: one value for each symbol
print('Is it raining on the dry day?', rain.evaluate(___))

**✅ You should see:**

```
Is it raining on the dry day? False
```

### A2 — Not, And, Or

Each connective keeps its parts and asks them for **their** values. That is the whole trick:
a big sentence is evaluated by evaluating its small pieces. Fill in the **three** blanks.

*Hint:* Python's `all(...)` is True when every item is True, and `any(...)` when at least one is.

In [ ]:
class Not(Sentence):
    def __init__(self, operand):
        self.operand = operand
    def evaluate(self, model):
        return ___  # blank 1: the opposite
    def formula(self):
        return '¬' + wrap(self.operand)
    def symbols(self):
        return self.operand.symbols()


class And(Sentence):
    def __init__(self, *conjuncts):
        self.conjuncts = list(conjuncts)
    def add(self, conjunct):                  # a KB grows one clue at a time
        self.conjuncts.append(conjunct)
    def evaluate(self, model):
        return ___(c.evaluate(model) for c in self.conjuncts)  # blank 2
    def formula(self):
        return ' ∧ '.join(wrap(c) for c in self.conjuncts)
    def symbols(self):
        return set().union(*(c.symbols() for c in self.conjuncts))


class Or(Sentence):
    def __init__(self, *disjuncts):
        self.disjuncts = list(disjuncts)
    def evaluate(self, model):
        return ___(d.evaluate(model) for d in self.disjuncts)  # blank 3
    def formula(self):
        return ' ∨ '.join(wrap(d) for d in self.disjuncts)
    def symbols(self):
        return set().union(*(d.symbols() for d in self.disjuncts))


wet = Symbol('wet')
model = {'rain': True, 'wet': False}
for s in [Not(rain), And(rain, wet), Or(rain, wet), Not(And(rain, wet))]:
    print(f'{s.formula():<14} is {s.evaluate(model)}')

**✅ You should see:**

```
¬rain          is False
rain ∧ wet     is False
rain ∨ wet     is True
¬(rain ∧ wet)  is True
```

### A3 — Implication and biconditional

Two facts you can turn straight into code:

* P ⇒ Q is false only when P is true and Q is false. So it means the same as **¬P ∨ Q**.
* P ⇔ Q is true when P and Q have the **same** value.

Fill in the **two** blanks.

In [ ]:
class Implication(Sentence):
    def __init__(self, antecedent, consequent):
        self.antecedent = antecedent     # the "if" part
        self.consequent = consequent     # the "then" part
    def evaluate(self, model):
        if_part = self.antecedent.evaluate(model)
        then_part = self.consequent.evaluate(model)
        return ___       # blank 1: same as ¬P ∨ Q
    def formula(self):
        return wrap(self.antecedent) + ' ⇒ ' + wrap(self.consequent)
    def symbols(self):
        return self.antecedent.symbols() | self.consequent.symbols()


class Biconditional(Sentence):
    def __init__(self, left, right):
        self.left = left
        self.right = right
    def evaluate(self, model):
        left = self.left.evaluate(model)
        right = self.right.evaluate(model)
        return ___          # blank 2: the same value?
    def formula(self):
        return wrap(self.left) + ' ⇔ ' + wrap(self.right)
    def symbols(self):
        return self.left.symbols() | self.right.symbols()

print('Library complete:', [c.__name__ for c in Sentence.__subclasses__()])

**✅ You should see:**

```
Library complete: ['Symbol', 'Not', 'And', 'Or', 'Implication', 'Biconditional']
```

### A4 — Truth tables, and the implication surprise

`truth_table` prints one row for every model of the symbols it finds, T for True, F for False.
Run it for the four connectives you already trust.

In [ ]:
def truth_table(*sentences):
    names = sorted(set().union(*(s.symbols() for s in sentences)))
    header = names + [s.formula() for s in sentences]
    widths = [max(5, len(h)) for h in header]
    print('  '.join(h.center(w) for h, w in zip(header, widths)))
    for values in itertools.product([True, False], repeat=len(names)):
        model = dict(zip(names, values))
        row = list(values) + [s.evaluate(model) for s in sentences]
        print('  '.join(('T' if v else 'F').center(w)
                        for v, w in zip(row, widths)))

P, Q = Symbol('P'), Symbol('Q')
truth_table(Not(P), And(P, Q), Or(P, Q))

**✅ You should see:**

```
  P      Q      ¬P   P ∧ Q  P ∨ Q
  T      T      F      T      T
  T      F      F      F      T
  F      T      T      F      T
  F      F      T      F      F
```

**Predict before you run.** Your teacher promises: *"If you score 90 or more (P), you will get
an A (Q)."* In which rows has the teacher **broken** the promise? Then fill in what you think
the column **P ⇒ Q** shows, top row first.

In [ ]:
# Rows in order: (P,Q) = (T,T), (T,F), (F,T), (F,F). Write True or False.
my_implication_column = [___, ___, ___, ___]

In [ ]:
# Run this to reveal the real answer.
truth_table(Implication(P, Q), Biconditional(P, Q))
real = [Implication(P, Q).evaluate({'P': p, 'Q': q})
        for p, q in itertools.product([True, False], repeat=2)]
print('your column:', my_implication_column)
print('real column:', real)

**✅ You should see:**

```
  P      Q    P ⇒ Q  P ⇔ Q
  T      T      T      T
  T      F      F      F
  F      T      T      F
  F      F      T      T
your column: [True, False, True, True]
real column: [True, False, True, True]
```

> ### Check yourself — Part A
>
> 1. In how many rows of its truth table is P ⇒ Q false?
>
> 2. Write P ⇒ Q using only ¬ and ∨.
>
> 3. Why does `And` need an `add()` method, but `Not` does not?

---
## Part B — The reasoning engine (4 marks)

> ### Remember
>
> • With **n** symbols there are **2ⁿ** models: each symbol is True or False, independently.
>
> • A **model of the KB** is a model in which the KB is true. Each clue throws some models away.
>
> • **KB ⊨ α** (the KB *entails* α) means α is true in **every** model of the KB. A single model where the KB is true and α is false (a **counter-example**) is enough to say *no*.
>
> • **Model checking** decides entailment by trying every model. It is simple and always right, but slow.

### B1 — Every possible world

`all_models` hands out every model of a set of symbol names, one at a time (it is a generator).

**Predict:** how many models do 3 symbols have? And 16 symbols?

In [ ]:
models_of_3  = ___
models_of_16 = ___

In [ ]:
def all_models(names):
    """Every model (dict name -> True/False) of the given symbol names."""
    names = sorted(names)
    for values in itertools.product([True, False], repeat=len(names)):
        yield dict(zip(names, values))

for m in all_models({'rain', 'hagrid', 'dumbledore'}):
    print(m)
print('3 symbols :', len(list(all_models('abc'))), '   you said', models_of_3)
print('16 symbols:', 2 ** 16, '   you said', models_of_16)

**✅ You should see:**

```
{'dumbledore': True, 'hagrid': True, 'rain': True}
{'dumbledore': True, 'hagrid': True, 'rain': False}
{'dumbledore': True, 'hagrid': False, 'rain': True}
{'dumbledore': True, 'hagrid': False, 'rain': False}
{'dumbledore': False, 'hagrid': True, 'rain': True}
{'dumbledore': False, 'hagrid': True, 'rain': False}
{'dumbledore': False, 'hagrid': False, 'rain': True}
{'dumbledore': False, 'hagrid': False, 'rain': False}
3 symbols : 8    you said 8
16 symbols: 65536    you said 65536
```

### B2 — Write down what we know

Case 1 arrives (it is the example in `harry.py`). Harry visits **either** Hagrid **or**
Dumbledore after school, never both. Here is what we know today:

1. If it did **not** rain, Harry visited Hagrid.
2. Harry visited Hagrid or Dumbledore.
3. He did **not** visit **both** of them.
4. Harry visited Dumbledore.

Translate each sentence into logic. Sentence 2 is done for you.

In [ ]:
rain       = Symbol('rain')        # It rained today.
hagrid     = Symbol('hagrid')      # Harry visited Hagrid today.
dumbledore = Symbol('dumbledore')  # Harry visited Dumbledore today.

knowledge = And(
    Implication(___, ___),   # 1. if it did not rain, Hagrid
    Or(hagrid, dumbledore),             # 2. Hagrid or Dumbledore
    Not(___),      # 3. not both
    ___,                        # 4. he visited Dumbledore
)
print(knowledge.formula())

**✅ You should see:**

```
(¬rain ⇒ hagrid) ∧ (hagrid ∨ dumbledore) ∧ ¬(hagrid ∧ dumbledore) ∧ dumbledore
```

Now mark the models of the KB. **Predict first:** in how many of the 8 possible worlds is every
one of the four clues true?

In [ ]:
kb_true_count = ___

In [ ]:
count = 0
for m in all_models(knowledge.symbols()):
    ok = knowledge.evaluate(m)
    count += ok
    print(m, '<-- the KB is true here' if ok else '')
print('models of the KB:', count, '   you said', kb_true_count)

**✅ You should see:**

```
{'dumbledore': True, 'hagrid': True, 'rain': True}
{'dumbledore': True, 'hagrid': True, 'rain': False}
{'dumbledore': True, 'hagrid': False, 'rain': True} <-- the KB is true here
{'dumbledore': True, 'hagrid': False, 'rain': False}
{'dumbledore': False, 'hagrid': True, 'rain': True}
{'dumbledore': False, 'hagrid': True, 'rain': False}
{'dumbledore': False, 'hagrid': False, 'rain': True}
{'dumbledore': False, 'hagrid': False, 'rain': False}
models of the KB: 1    you said 1
```

### B3 — Model checking

Now turn that idea into a function. `model_check(kb, query)` looks for a **counter-example**:
a model where the KB is true but the query is false. If it finds one, the KB does **not** entail
the query. If no such model exists, it does. Fill in the **three** blanks.

In [ ]:
def model_check(kb, query):
    """True if the knowledge base entails the query (KB ⊨ query)."""
    names = kb.symbols() | query.symbols()
    for model in all_models(names):
        if ___ and ___:  # blanks 1, 2
            return ___        # blank 3: we found a counter-example
    return True

print('Does the KB entail rain?      ', model_check(knowledge, rain))
print('Does the KB entail hagrid?    ', model_check(knowledge, hagrid))
print('Does the KB entail not hagrid?', model_check(knowledge, Not(hagrid)))

**✅ You should see:**

```
Does the KB entail rain?       True
Does the KB entail hagrid?     False
Does the KB entail not hagrid? True
```

> ### How does `logic.py` do it?
>
> The original `model_check` in `src/logic.py` builds the models **recursively**: it picks one symbol, makes one copy of the model with it True and one with it False, and recurses until every symbol has a value. It visits the same 2ⁿ models as your loop, in a different order. Your version uses `itertools.product` to list them, which is easier to read.

### B4 — YES, NO or MAYBE

A detective has three possible answers, not two. If the KB entails the query, the answer is
YES. If it entails **not** the query, the answer is NO. If it entails neither, we do not know
yet: MAYBE. Fill in the **two** blanks.

In [ ]:
def ask(kb, query):
    if model_check(kb, query):
        return 'YES'
    if model_check(kb, ___):     # blank 1: is the query surely false?
        return 'NO'
    return ___                        # blank 2: neither is proved

for q in [rain, hagrid, dumbledore]:
    print(f'{q.formula():<11}', ask(knowledge, q))

**✅ You should see:**

```
rain        YES
hagrid      NO
dumbledore  YES
```

> ### Check yourself — Part B
>
> 1. A KB has 5 symbols. How many models does `model_check` try, at most?
>
> 2. `model_check(kb, q)` returns False. Does that mean q is false? Explain.
>
> 3. Why does `model_check` stop at the **first** counter-example instead of trying the rest?

---
## Part C — Case 1: Did it rain? (2 marks)

> ### Remember
>
> • **More knowledge means fewer models.** Each clue can only throw worlds away, never add them.
>
> • When the KB has **no** models at all, it is **contradictory**. Then every query is "entailed", because there is no world left to be a counter-example.

### C1 — Solve the case

You already have everything you need.

In [ ]:
print('Did it rain today?', ask(knowledge, ___))

**✅ You should see:**

```
Did it rain today? YES
```

### C2 — A witness is less sure

Suppose we never learned clue 4. We only know clues 1–3. **Predict:** what will the detective
say about `rain` now: YES, NO or MAYBE?

In [ ]:
prediction_less_knowledge = ___

In [ ]:
less = And(
    Implication(Not(rain), hagrid),
    Or(hagrid, dumbledore),
    Not(And(hagrid, dumbledore)),
)
print('rain      :', ask(less, rain), '   you said', prediction_less_knowledge)
print('models    :', sum(less.evaluate(m) for m in all_models(less.symbols())))

**✅ You should see:**

```
rain      : MAYBE    you said MAYBE
models    : 3
```

### C3 — A lying witness

Now someone adds a fifth clue: *"It definitely did not rain today."* This contradicts what we
already know. **Predict:** what will `ask` say about `rain` now? And about `Not(rain)`?

In [ ]:
prediction_rain     = ___
prediction_not_rain = ___

In [ ]:
liar_kb = And(*knowledge.conjuncts, Not(rain))
print('models of this KB :', sum(liar_kb.evaluate(m)
                                 for m in all_models(liar_kb.symbols())))
print('rain              :', ask(liar_kb, rain))
print('not rain          :', ask(liar_kb, Not(rain)))
print('you said          :', prediction_rain, prediction_not_rain)

**✅ You should see:**

```
models of this KB : 0
rain              : YES
not rain          : YES
you said          : YES YES
```

**Your turn: fix the engine.** Before answering, `ask` should check that at least one model of
the KB exists. `satisfiable` is True as soon as it finds one. Fill in the **two** blanks.

In [ ]:
def satisfiable(kb):
    """True if at least one model makes the KB true."""
    return ___(kb.evaluate(m) for m in all_models(kb.symbols()))

def ask(kb, query):
    if not ___:
        return 'CONTRADICTION'
    if model_check(kb, query):
        return 'YES'
    if model_check(kb, Not(query)):
        return 'NO'
    return 'MAYBE'

print('honest KB, rain  :', ask(knowledge, rain))
print('liar KB, rain    :', ask(liar_kb, rain))

**✅ You should see:**

```
honest KB, rain  : YES
liar KB, rain    : CONTRADICTION
```

> ### Check yourself — Part C
>
> 1. Adding a clue to a KB can never increase the number of its models. Why not?
>
> 2. A KB entails both q and ¬q. What do you know about the KB?

---
## Part D — Case 2: The Clue mystery (2 marks)

> ### Remember
>
> • In the board game *Clue* (also sold as *Cluedo*), one person, one room and one weapon are hidden in an envelope. Every card you **see** is **not** in the envelope.
>
> • "At least one of these is true" is an **Or**. "This one is false" is a **Not**.
>
> • You can ask the KB about any symbol, at any time, and watch the answers change as clues arrive.

This is the case in `clue.py`. There are three suspects, three rooms and three weapons. The
board below shows what the detective knows about each: ✅ surely in the envelope, ❌ surely
not, ❓ not known yet.

In [ ]:
mustard, plum, scarlet = (Symbol('ColMustard'), Symbol('ProfPlum'),
                          Symbol('MsScarlet'))
ballroom, kitchen, library = (Symbol('ballroom'), Symbol('kitchen'),
                              Symbol('library'))
knife, revolver, wrench = (Symbol('knife'), Symbol('revolver'),
                           Symbol('wrench'))
groups = {'who':   [mustard, plum, scarlet],
          'where': [ballroom, kitchen, library],
          'how':   [knife, revolver, wrench]}

MARK = {'YES': '✅', 'NO': '❌', 'MAYBE': '❓', 'CONTRADICTION': '💥'}

def board(kb, groups):
    for title, group in groups.items():
        cells = [f'{MARK[ask(kb, s)]} {s.formula():<10}' for s in group]
        print(f'{title:<6}', '  '.join(cells))

### D1 — The rules of the game

The envelope holds a person, a room and a weapon. Write that as three `Or`s.

In [ ]:
clue_kb = And(
    Or(mustard, plum, scarlet),      # there is a murderer
    ___,  # there is a room
    ___,     # there is a weapon
)
board(clue_kb, groups)

**✅ You should see:**

```
who    ❓ ColMustard  ❓ ProfPlum    ❓ MsScarlet
where  ❓ ballroom    ❓ kitchen     ❓ library
how    ❓ knife       ❓ revolver    ❓ wrench
```

### D2 — The clues arrive, one by one

**Clue 1.** You were dealt three cards: Colonel Mustard, the kitchen and the revolver. None of
them can be in the envelope.

In [ ]:
clue_kb.add(And(Not(mustard), Not(___), Not(___)))
board(clue_kb, groups)

**✅ You should see:**

```
who    ❌ ColMustard  ❓ ProfPlum    ❓ MsScarlet
where  ❓ ballroom    ❌ kitchen     ❓ library
how    ❓ knife       ❌ revolver    ❓ wrench
```

**Clue 2.** You suggested *"Ms Scarlet, in the library, with the wrench"*. Another player
showed you one of those three cards, secretly, so you do not know which. At least one of the
three is **not** in the envelope.

In [ ]:
clue_kb.add(Or(Not(scarlet), Not(library), ___))
board(clue_kb, groups)

**✅ You should see:**

```
who    ❌ ColMustard  ❓ ProfPlum    ❓ MsScarlet
where  ❓ ballroom    ❌ kitchen     ❓ library
how    ❓ knife       ❌ revolver    ❓ wrench
```

Nothing changed on the board, but the KB did learn something. Hold on to that thought.

**Clue 3.** You see the Professor Plum card.

In [ ]:
clue_kb.add(___)
board(clue_kb, groups)

**✅ You should see:**

```
who    ❌ ColMustard  ❌ ProfPlum    ✅ MsScarlet
where  ❓ ballroom    ❌ kitchen     ❓ library
how    ❓ knife       ❌ revolver    ❓ wrench
```

### D3 — Crack the case

**Clue 4.** You see the ballroom card. **Predict:** after this clue, which weapon will the board
show as ✅? Write `'knife'`, `'wrench'` or `'still unknown'`.

In [ ]:
prediction_weapon = ___

In [ ]:
clue_kb.add(Not(ballroom))
board(clue_kb, groups)
print('you said:', prediction_weapon)

**✅ You should see:**

```
who    ❌ ColMustard  ❌ ProfPlum    ✅ MsScarlet
where  ❌ ballroom    ❌ kitchen     ✅ library
how    ✅ knife       ❌ revolver    ❌ wrench
you said: knife
```

> ### Check yourself — Part D
>
> 1. Why is clue 2 an `Or` of three `Not`s, and not an `And`?
>
> 2. The KB has 9 symbols. How many models does `model_check` try for each question?

---
## Part E — Case 3: The sorting puzzle (2 marks)

> ### Remember
>
> • Logic has no numbers and no "exactly one". You build them out of the connectives: "at least one" is an **Or**, and "at most one" is a set of rules "if this, then not that".
>
> • Writing the rules **with loops** turns a 100-rule KB into a few lines of code.

This is the case in `puzzle.py`. Four people must each be sorted into a **different** one of
four houses. A symbol such as `MinervaGryffindor` means "Minerva is in Gryffindor". That makes
4 × 4 = 16 symbols.

### E1 — The rules of sorting

Fill in the **three** blanks. Each rule's English meaning is in the comment above it.

In [ ]:
people = ['Gilderoy', 'Pomona', 'Minerva', 'Horace']
houses = ['Gryffindor', 'Hufflepuff', 'Ravenclaw', 'Slytherin']

def S(person, house):
    return Symbol(person + house)

sorting_rules = And()

# Rule 1: each person belongs to a house.
for person in people:
    sorting_rules.add(Or(*[S(person, h) for h in ___]))

# Rule 2: only one house per person.
for person in people:
    for h1 in houses:
        for h2 in houses:
            if h1 != h2:
                sorting_rules.add(Implication(S(person, h1),
                                              ___))

# Rule 3: only one person per house.
for house in houses:
    for p1 in people:
        for p2 in people:
            if ___:
                sorting_rules.add(Implication(S(p1, house),
                                              Not(S(p2, house))))

print('rules  :', len(sorting_rules.conjuncts))
print('symbols:', len(sorting_rules.symbols()))

**✅ You should see:**

```
rules  : 100
symbols: 16
```

**Predict:** before any clue, how many models do these rules have? That is, in how many ways
can 4 people go into 4 houses, one person per house?

In [ ]:
prediction_sortings = ___

In [ ]:
def count_models(kb):
    return sum(kb.evaluate(m) for m in all_models(kb.symbols()))

print('sortings allowed by the rules:', count_models(sorting_rules),
      '   you said', prediction_sortings)

**✅ You should see:**

```
sortings allowed by the rules: 24    you said 24
```

### E2 — The clues, and the solution

1. Gilderoy is in Gryffindor or Ravenclaw.
2. Pomona is **not** in Slytherin.
3. Minerva is in Gryffindor.

Fill in the blanks, then run. ⏳ **This cell takes about 10 seconds.** That slowness is the
subject of Part G.

In [ ]:
puzzle_clues = [
    Or(S('Gilderoy', 'Gryffindor'), ___),  # clue 1
    ___,                                # clue 2
    S('Minerva', 'Gryffindor'),                                    # clue 3
]
sorting_kb = And(*sorting_rules.conjuncts, *puzzle_clues)

start = time.perf_counter()
for person in people:
    for house in houses:
        if model_check(sorting_kb, S(person, house)):
            print(person, '->', house)
print(f'time taken ≈ {time.perf_counter() - start:.1f} seconds')

**✅ You should see:**

```
Gilderoy -> Ravenclaw
Pomona -> Hufflepuff
Minerva -> Gryffindor
Horace -> Slytherin
time taken ≈ (yours will differ: write it in the observations table)
```

### E3 — Was every clue needed?

A good detective asks whether a clue really mattered. The cell below removes each clue in turn
and counts the sortings that are then allowed. **Predict:** which clue, if removed, would leave
the **most** possible sortings? Write 1, 2 or 3. (⏳ about 5 seconds.)

In [ ]:
prediction_most_important = ___

In [ ]:
for i in range(len(puzzle_clues)):
    others = [c for j, c in enumerate(puzzle_clues) if j != i]
    kb = And(*sorting_rules.conjuncts, *others)
    print(f'without clue {i + 1}: {count_models(kb)} sortings')
print('you said clue', prediction_most_important)

**✅ You should see:**

```
without clue 1: 4 sortings
without clue 2: 2 sortings
without clue 3: 8 sortings
you said clue 3
```

> ### Check yourself — Part E
>
> 1. Rule 2 adds 4 × 4 × 3 = 48 implications. What does each one say in English?
>
> 2. Why does the puzzle use 16 symbols, rather than 4 symbols each holding a house name?

---
## Part F — Case 4: Mastermind (2 marks)

> ### Remember
>
> • In this version of **Mastermind**, the secret code is the four colours red, blue, green and yellow, each used once, in some order. After each guess you are told **how many** colours are in the right place, but not which ones.
>
> • "Exactly k of these are true" is an **Or** over every way of choosing which k are true. With `itertools.combinations` you can write it once and use it everywhere.

This is the case in `mastermind.py`. A symbol such as `red0` means "red is in position 0".

### F1 — A tool for "exactly k"

`mastermind.py` writes "exactly 2 of 4" by hand, as six cases. Instead you will write a function
that builds the cases for any k. For each choice of k sentences to be true, it makes an `And`
of those, plus `Not` of all the others. Fill in the **two** blanks.

In [ ]:
def exactly(k, sentences):
    """A sentence that is true when exactly k of the sentences are true."""
    options = []
    for chosen in itertools.combinations(sentences, ___):
        rest = [s for s in sentences if s not in chosen]
        options.append(And(*chosen, *[___ for s in rest]))
    return Or(*options)

a, b, c = Symbol('a'), Symbol('b'), Symbol('c')
print(exactly(1, [a, b, c]).formula())
print('models of "exactly 2 of 4":',
      count_models(exactly(2, [a, b, c, Symbol('d')])))

**✅ You should see:**

```
(a ∧ ¬b ∧ ¬c) ∨ (b ∧ ¬a ∧ ¬c) ∨ (c ∧ ¬a ∧ ¬b)
models of "exactly 2 of 4": 6
```

### F2 — The rules of Mastermind, in eight lines

Every colour is in exactly one position, and every position holds exactly one colour. With
`exactly`, that takes two loops. Fill in the **two** blanks.

In [ ]:
colours = ['red', 'blue', 'green', 'yellow']

def at(colour, position):
    return Symbol(f'{colour}{position}')

mm_rules = And()
for colour in colours:            # each colour: exactly one place
    mm_rules.add(exactly(1, [at(colour, i) for i in range(4)]))
for i in range(4):                # each place: exactly one colour
    mm_rules.add(exactly(___, [at(colour, ___) for colour in colours]))

n_rules, n_symbols = len(mm_rules.conjuncts), len(mm_rules.symbols())
print('rules  :', n_rules, '   symbols:', n_symbols)
print('possible secret codes:', count_models(mm_rules))

**✅ You should see:**

```
rules  : 8    symbols: 16
possible secret codes: 24
```

### F3 — Two guesses, one answer

| Guess | Position 0 | Position 1 | Position 2 | Position 3 | Feedback |
|---|---|---|---|---|---|
| 1 | red | blue | green | yellow | **2** in the right place |
| 2 | blue | red | green | yellow | **0** in the right place |

Write each guess's feedback as a clue with `exactly`, then solve. (⏳ about 10 seconds.)

In [ ]:
guess1 = [at('red', 0), at('blue', 1), at('green', 2), at('yellow', 3)]
guess2 = [at('blue', 0), at('red', 1), at('green', 2), at('yellow', 3)]

mm_kb = And(*mm_rules.conjuncts,
            exactly(___, guess1),
            exactly(___, ___))

start = time.perf_counter()
answer = [colour for i in range(4) for colour in colours
          if model_check(mm_kb, at(colour, i))]
print('the secret code is:', answer)
print(f'time taken ≈ {time.perf_counter() - start:.1f} seconds')

**✅ You should see:**

```
the secret code is: ['red', 'blue', 'yellow', 'green']
time taken ≈ (yours will differ: write it in the observations table)
```

> ### Check yourself — Part F
>
> 1. How many `And`s does `exactly(2, ...)` build for 4 sentences? For 6 sentences?
>
> 2. What does `exactly(0, [p, q])` mean, in plain logic?

---
## Part G — The engine is too slow (2 marks)

> ### Remember
>
> • To answer 16 questions about the sorting puzzle, `model_check` walked through 65,536 models **for each question**. That is about a million evaluations of a 103-clue KB.
>
> • But the models of the KB **do not change** between questions. List them once, keep them, and answer every question from that short list.

### G1 — List the models once

Fill in the **three** blanks.

In [ ]:
def kb_models(kb):
    """Every model of the KB, as a list. The slow part, done once."""
    return [m for m in all_models(kb.symbols()) if ___]

def ask_fast(models, query):
    """Answer a query from a list of the KB's models."""
    values = {query.evaluate(m) for m in models}  # the values query takes
    if not models:
        return 'CONTRADICTION'
    if values == ___:            # true in every model of the KB
        return 'YES'
    if values == ___:           # false in every model of the KB
        return 'NO'
    return 'MAYBE'

start = time.perf_counter()
sort_models = kb_models(sorting_kb)          # the slow part, once
print('models of the sorting KB:', len(sort_models))
for person in people:
    row = [f'{MARK[ask_fast(sort_models, S(person, h))]} {h:<10}'
           for h in houses]
    print(f'{person:<9}', '  '.join(row))
print(f'time taken ≈ {time.perf_counter() - start:.1f} seconds')

**✅ You should see:**

```
models of the sorting KB: 1
Gilderoy  ❌ Gryffindor  ❌ Hufflepuff  ✅ Ravenclaw   ❌ Slytherin
Pomona    ❌ Gryffindor  ✅ Hufflepuff  ❌ Ravenclaw   ❌ Slytherin
Minerva   ✅ Gryffindor  ❌ Hufflepuff  ❌ Ravenclaw   ❌ Slytherin
Horace    ❌ Gryffindor  ❌ Hufflepuff  ❌ Ravenclaw   ✅ Slytherin
time taken ≈ (yours will differ: write it in the observations table)
```

The fast engine answered **all 16** questions in about the time the slow one took for **one**,
and it also filled in every ❌, which the slow loop never printed. Now Mastermind:

In [ ]:
start = time.perf_counter()
mm_models = kb_models(mm_kb)
code_found = [colour for i in range(4) for colour in colours
              if ask_fast(mm_models, at(colour, i)) == 'YES']
print('models:', len(mm_models), '   code:', code_found)
print(f'time taken ≈ {time.perf_counter() - start:.1f} seconds')

**✅ You should see:**

```
models: 1    code: ['red', 'blue', 'yellow', 'green']
time taken ≈ (yours will differ: write it in the observations table)
```

### G2 — Why model checking cannot grow up

Listing models once fixed the repeated work, but the first list still costs 2ⁿ. **Predict:** if
your computer checks one million models per second, roughly how long would 50 symbols take?
Write `'seconds'`, `'days'` or `'years'`.

In [ ]:
prediction_50_symbols = ___

In [ ]:
for n in [3, 9, 16, 30, 50, 100]:
    seconds = 2 ** n / 1_000_000
    years = seconds / (60 * 60 * 24 * 365)
    print(f'{n:>3} symbols: {2 ** n:>32,} models  ≈ {seconds:>10.3g} s'
          f' = {years:.3g} years')
print('you said:', prediction_50_symbols)

**✅ You should see:**

```
  3 symbols:                                8 models  ≈      8e-06 s = 2.54e-13 years
  9 symbols:                              512 models  ≈   0.000512 s = 1.62e-11 years
 16 symbols:                           65,536 models  ≈     0.0655 s = 2.08e-09 years
 30 symbols:                    1,073,741,824 models  ≈       1.07e+03 s = 3.4e-05 years
 50 symbols:            1,125,899,906,842,624 models  ≈   1.13e+09 s = 35.7 years
100 symbols: 1,267,650,600,228,229,401,496,703,205,376 models  ≈   1.27e+24 s = 4.02e+16 years
you said: years
```

> ### Check yourself — Part G
>
> 1. The slow loop in E2 asked 16 questions. Roughly how many models did it evaluate in all?
>
> 2. `kb_models` returns an empty list. What should the detective conclude?

---
## Part H — Final case: the island of truth-tellers and liars (2 marks)

> ### Remember
>
> • On this island every person is a **knight**, who always tells the truth, or a **liar**, who always lies. Nobody is both.
>
> • So whatever a person says is true **exactly when** that person is a knight. That is a **biconditional**: `Biconditional(knight, statement)`.

### H1 — The two tools you need

Fill in the **two** blanks in `says`.

In [ ]:
def islander(name):
    """A person's two symbols, and the rule that they are exactly one kind."""
    knight, liar = Symbol(name + 'Knight'), Symbol(name + 'Liar')
    return knight, liar, exactly(1, [knight, liar])

def says(speaker_is_knight, statement):
    """The statement is true if and only if the speaker is a knight."""
    return Biconditional(___, ___)

def solve(kb, people):
    """Print each person's kind, or CONTRADICTION if nobody could say that."""
    models = kb_models(kb)
    if not models:
        print('💥 CONTRADICTION: this conversation could not happen.')
        return
    for name, knight in people.items():
        verdict = ask_fast(models, knight)
        kind = {'YES': 'knight', 'NO': 'liar'}.get(verdict, 'unknown')
        print(f'{name} is a {kind}')

ali_k, ali_l, ali_rule = islander('Ali')
sara_k, sara_l, sara_rule = islander('Sara')
zain_k, zain_l, zain_rule = islander('Zain')
print(says(ali_k, sara_l).formula())

**✅ You should see:**

```
AliKnight ⇔ SaraLiar
```

### H2 — Two islanders

**Puzzle 1.** Ali says: *"At least one of us is a liar."* (Sara says nothing.)

In [ ]:
puzzle1 = And(ali_rule, sara_rule,
              says(ali_k, ___))
solve(puzzle1, {'Ali': ali_k, 'Sara': sara_k})

**✅ You should see:**

```
Ali is a knight
Sara is a liar
```

**Puzzle 2.** Ali says: *"If I am a knight, then Sara is a knight too."* **Predict** each
person's kind, then encode the statement with `Implication` and run.

In [ ]:
prediction_puzzle2 = {'Ali': ___, 'Sara': ___}

In [ ]:
puzzle2 = And(ali_rule, sara_rule,
              says(ali_k, ___))
solve(puzzle2, {'Ali': ali_k, 'Sara': sara_k})
print('you said:', prediction_puzzle2)

**✅ You should see:**

```
Ali is a knight
Sara is a knight
you said: {'Ali': 'knight', 'Sara': 'knight'}
```

### H3 — Three islanders

**Puzzle 3.** Ali says: *"Zain is a liar."* Sara says: *"Ali and Zain are the same kind."*
Zain says: *"Sara is a knight."*

*Hint:* "Ali and Zain are the same kind" means Ali is a knight if and only if Zain is a knight.

In [ ]:
puzzle3 = And(ali_rule, sara_rule, zain_rule,
              says(ali_k, ___),
              says(sara_k, ___),
              says(___, sara_k))
solve(puzzle3, {'Ali': ali_k, 'Sara': sara_k, 'Zain': zain_k})

**✅ You should see:**

```
Ali is a knight
Sara is a liar
Zain is a liar
```

### H4 — The impossible conversation

**Puzzle 4.** Ali says: *"Sara is a knight."* Sara says: *"Ali is a liar."* **Predict:** who is
the knight?

In [ ]:
prediction_puzzle4 = ___

In [ ]:
puzzle4 = And(ali_rule, sara_rule,
              says(ali_k, sara_k),
              says(sara_k, ali_l))
solve(puzzle4, {'Ali': ali_k, 'Sara': sara_k})
print('you said:', prediction_puzzle4)

**✅ You should see:**

```
💥 CONTRADICTION: this conversation could not happen.
you said: nobody: it cannot happen
```

### H5 — Your own puzzle (not marked, but the best part)

Invent a puzzle with two or three islanders that has **exactly one** solution. Write it in the
cell below, check that the engine solves it, then give it to the classmate next to you to solve
on paper. If your engine says CONTRADICTION or *unknown*, change a statement and try again.

In [ ]:
# My puzzle, in English:
#   ...
my_puzzle = And(ali_rule, sara_rule,
                says(ali_k, ___))
solve(my_puzzle, {'Ali': ali_k, 'Sara': sara_k})

> ### Check yourself — Part H
>
> 1. Why is a statement modelled with ⇔ and not with ⇒?
>
> 2. Could an islander ever say *"I am a liar"*? Use the engine's idea of a contradiction.

---
## Part I — Bonus: an agent that plays Mastermind (not marked)

AIMA describes a **knowledge-based agent** with two operations. It **TELL**s its knowledge base
what it has just seen, and it **ASK**s the knowledge base what to do. Here is one that plays
Mastermind against a secret code, using only things you built today:

* **ASK:** pick any code that is still a model of the KB, and guess it.
* **TELL:** turn the feedback into a clue with `exactly(k, guess)`, and keep only the models
  that agree with it.

Run it. Then change `secret` to any order of the four colours, and run it again.

In [ ]:
secret = ['yellow', 'red', 'blue', 'green']     # try your own order

def feedback(secret, guess):
    return sum(s == g for s, g in zip(secret, guess))

possible = kb_models(mm_rules)            # all 24 codes are possible at first
for turn in range(1, 10):
    model = possible[0]                   # ASK: any code still possible
    guess = [c for i in range(4) for c in colours if model[f'{c}{i}']]
    k = feedback(secret, guess)
    print(f'turn {turn}: {len(possible):>2} codes possible, '
          f'guess {guess} -> {k} right')
    if k == 4:
        print('Cracked it in', turn, 'guesses.')
        break
    clue = exactly(k, [at(c, i) for i, c in enumerate(guess)])   # TELL
    possible = [m for m in possible if clue.evaluate(m)]

**✅ You should see:**

```
turn 1: 24 codes possible, guess ['blue', 'green', 'red', 'yellow'] -> 0 right
turn 2:  9 codes possible, guess ['green', 'blue', 'yellow', 'red'] -> 0 right
turn 3:  4 codes possible, guess ['red', 'yellow', 'blue', 'green'] -> 2 right
turn 4:  2 codes possible, guess ['yellow', 'red', 'blue', 'green'] -> 4 right
Cracked it in 4 guesses.
```

---
## One-page summary: the line that matters

| Idea | The line of code that carries it |
|---|---|
| a model | `{'rain': True, 'hagrid': False}`: one value per symbol |
| evaluate a sentence | each connective asks its parts: `all(...)` for ∧, `any(...)` for ∨ |
| implication | `not P or Q`: false only when P is true and Q is false |
| biconditional | `P == Q`: the same value |
| every possible world | `itertools.product([True, False], repeat=n)` gives 2ⁿ models |
| entailment | `if kb.evaluate(model) and not query.evaluate(model): return False` |
| YES / NO / MAYBE | entails q → YES; entails ¬q → NO; neither → MAYBE |
| contradiction | no model at all: `not satisfiable(kb)` |
| exactly k | `Or` over `itertools.combinations(sentences, k)` |
| faster answers | list the KB's models once: `kb_models(kb)` |
| truth-teller puzzles | `Biconditional(speaker_is_knight, statement)` |

## Observations

Fill these in from **your own** output.

| Part | Quantity | Your value |
|---|---|---|
| B | models of the Case 1 KB, out of 8 |  |
| C | models when clue 4 is removed / when the lying clue is added |  |
| D | the clue after which the murderer was known |  |
| D | the solution of the Clue case |  |
| E | sortings allowed by the rules alone / with all three clues |  |
| E | time taken by the slow loop in E2 (seconds) |  |
| F | the secret code / time taken by the slow loop (seconds) |  |
| G | time taken by the fast engine for the sorting puzzle (seconds) |  |
| G | how many times faster than E2 |  |
| H | the solution of puzzle 3 |  |
| I | guesses the agent needed for your own secret code |  |

## Discussion questions

Answer each in two or three sentences, in the cell below.

1. Model checking never makes a mistake, but it is rarely used on big problems. Why?
2. In Case 2, clue 2 changed nothing on the board when it arrived. Was it useless? Explain with
   what happened after clue 4.
3. Our engine reports CONTRADICTION. Give one real situation (not from this lab) where finding
   that a set of facts is contradictory is itself the useful answer.
4. `puzzle.py` needs 100 rules; your Mastermind rules needed 8. Do the two KBs say different
   things? What did `exactly` change?

*Your answers:*

## Viva questions

Answer each in one or two sentences.

1. What is a model? How many does a KB with n symbols have to check?
2. State the meaning of KB ⊨ α in one sentence.
3. In which single case is P ⇒ Q false?
4. Point to the line in `model_check` that finds a counter-example.
5. What is the difference between "the engine said NO" and "`model_check` returned False"?
6. Why does a contradictory KB entail everything?
7. How would you write "at most one of p, q, r is true" in logic?
8. What did `kb_models` change about the cost of answering 16 questions?
9. Why is a truth-teller's statement written with ⇔?
10. Name the two operations of a knowledge-based agent, and say where each happens in Part I.

## Submission checklist

- [ ] The notebook runs top to bottom from a clean kernel (Kernel → Restart & Run All)
- [ ] Every `___` blank is filled, and your output matches every **✅ You should see** box
- [ ] Every prediction cell holds your guess **from before** you ran the reveal
- [ ] You invented and solved your own island puzzle in H5
- [ ] The observations table is filled in from your own output, including your timings
- [ ] The discussion questions are answered

In [ ]:
NAME = ""
ROLL_NUMBER = ""
print(NAME, ROLL_NUMBER)